# F6 hand recomputation of three assets (plan F6 step 6, FR7)

Written blind: the F6 loss code (`src/flood_lab/risk/`, `checks/f6_risk.py`, `tests/flood/test_f6_risk.py`,
`assets/exposures.py`, `src/crc_event_loss/`) was not opened. Inputs read: `loss-rules.md` (rules FF, BD, C, V, A),
`curves.yaml` (points), `catalogue.py` docstring (strata, block weights, `WET_M`), `f4/selection.json`,
the 150 per-run files `f4/frequency_cover/per_run/*.npz`, `site/flood/data/exposures.csv`,
`assets_oed.csv`, `buildings_detail.csv`, and CRC's public docs (tabulated impacts interpolate linearly
between knots and refuse values outside them; `Distribution.quantile(q)` is the smallest loss whose
cumulative probability reaches q).

Assets: substation **S-383102724** (Hudson Avenue), building **B-3000013**, building **B-3000650**
(substitute value, basement Y, read at −4 ft).

Everything below is numpy/pandas and linear interpolation written out here. Only the footprint order
(which index of `walls` belongs to a building) uses geopandas, repeating F3's `study_footprints`
selection (representative point inside the study area, file order).

Order of the notebook: inputs, per-run depths, ratios and losses, **results table (section 6) written
before any builder output is read**, then the comparison (section 7).

In [1]:
import glob, json
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
F4 = ROOT / "data/cache/flood/f4"
WET_M = 0.05          # catalogue.WET_M: the builder's stated main-case cut (a [P] choice, pending Yuri)
M = 1000              # synthetic years
FT = 0.3048
pd.set_option("display.width", 200, "display.max_columns", 30)

## 1. Runs and year weights (F4 block weights)

In [2]:
sel = json.load(open(F4 / "selection.json"))
runs = pd.DataFrame(sel)[["label", "rank", "stratum", "block_first", "block_last", "weight"]]
# block weight = number of years in the block (catalogue docstring); check it against the stated weight
assert (runs.block_last - runs.block_first + 1 == runs.weight).all()
assert len(runs) == 150 and runs.weight.sum() == M
runs = runs.sort_values("rank").reset_index(drop=True)
print(runs.groupby("stratum").weight.agg(["count", "sum"]))

         count    sum
stratum              
T1          30   30.0
T2          60  120.0
T3          60  850.0


## 2. Which index of `walls` is each building (F3 footprint order)

In [3]:
import geopandas as gpd
fp = gpd.read_file(ROOT / "data/cache/flood/nyc-building-footprints/footprints-box.geojson").to_crs("EPSG:32618")
fp = fp[fp.geometry.notna() & ~fp.geometry.is_empty]
study = gpd.read_file(ROOT / "site/flood/data/study_area.geojson").to_crs("EPSG:32618").geometry.union_all()
fp = fp[fp.geometry.representative_point().within(study).to_numpy()].reset_index(drop=True)
bd = pd.read_csv(ROOT / "data/cache/flood/assets/buildings_detail.csv")
doitt = bd.set_index("LocNumber").detail_doitt_id
ids = fp.doitt_id.astype(int).to_numpy()
print("study footprints:", len(fp))
WALL_INDEX = {}
for loc in ["B-3000013", "B-3000650"]:
    hit = np.flatnonzero(ids == doitt[loc])
    assert len(hit) == 1
    WALL_INDEX[loc] = int(hit[0])
print(WALL_INDEX)

study footprints: 17750
{'B-3000013': 16361, 'B-3000650': 2533}


## 3. Depths in the 150 runs

In [4]:
SITE = "S-383102724"
rows = []
for r in runs.itertuples():
    d = np.load(F4 / "frequency_cover/per_run" / f"{r.label}.npz")
    assert d["walls"].shape == (len(fp),)
    sites = list(d["site_ids"])
    rows.append({
        "label": r.label, "weight": r.weight,
        SITE: float(d["site_lead"][sites.index(SITE)]),
        "B-3000013": float(d["walls"][WALL_INDEX["B-3000013"]]),
        "B-3000650": float(d["walls"][WALL_INDEX["B-3000650"]]),
    })
depth = pd.DataFrame(rows)
assert depth.notna().all().all()
print(depth.describe().T[["min", "50%", "max"]])

             min  50%        max
weight       1.0  2.0  24.000000
S-383102724  0.0  0.0   0.504000
B-3000013    0.0  0.0   0.163927
B-3000650    0.0  0.0   0.438779


## 4. Assets: value, first-floor height, basement, curve

Mid values are the 2026-dollar mid values of `exposures.csv` (V-1 to V-6 already applied there; checked
here only that the substation's is the asset table's mid × 1.060723 and B-3000013's is NSI × the 4Q25→4Q26 factor
is not re-derived).

In [5]:
ex = pd.read_csv(ROOT / "site/flood/data/exposures.csv").set_index("LocNumber")
oed = pd.read_csv(ROOT / "site/flood/data/assets_oed.csv").set_index("LocNumber")
A = ex.loc[[SITE, "B-3000013", "B-3000650"], ["occupancy", "storeys", "basement", "ffh_m", "value_mid_usd_2026", "curve_id", "value_source_quarter"]]
print(A.to_string())
print("substation factor 4Q24->4Q26:", ex.loc[SITE, "value_mid_usd_2026"] / oed.loc[SITE, "lab_value_mid_usd"])
print("OSM role:", oed.loc[SITE, "lab_substation_kind"], oed.loc[SITE, "lab_voltage_kv"], "kV")

            occupancy  storeys basement  ffh_m  value_mid_usd_2026                                                                         curve_id value_source_quarter
LocNumber                                                                                                                                                               
S-383102724       NaN      NaN      NaN    NaN        6.237558e+07  nirandjan_F2|williams_substation_max|williams_substation_min|bor_hv345_anchored                 4Q24
B-3000013       RES3D      1.0        N  0.160        1.001436e+06                                                                 hazus_struct_204                 4Q25
B-3000650        RES1      4.0        Y  3.844        5.572168e+05                                                                 hazus_struct_110                 4Q22
substation factor 4Q24->4Q26: 1.0607232303704028
OSM role: distribution 138.0 kV


/var/folders/_x/xxnz7zw16c315zf2q_vw4h1c0000gn/T/ipykernel_83634/841402693.py:2: DtypeWarning: Columns (3: LocName, 33: lab_osm_ids, 36: lab_substation_kind, 37: lab_indoor, 38: lab_reviewed) have mixed types. Specify dtype option on import or set low_memory=False.
  oed = pd.read_csv(ROOT / "site/flood/data/assets_oed.csv").set_index("LocNumber")


## 5. Curves and the reading written out

`interp(x, xs, ys)`: find the segment `xs[i] <= x <= xs[i+1]` and return `ys[i] + (x - xs[i]) (ys[i+1] - ys[i]) / (xs[i+1] - xs[i])`;
outside `[xs[0], xs[-1]]` it raises (CRC refuses out-of-domain values, no extrapolation).

Substation families (C-3, C-5 role basis: Hudson Avenue is `substation=distribution`, so Williams
"substation"): `low` = nirandjan_F2, `high` = williams_substation_max, `williams_min` = williams_substation_min,
`bor` = bor_hv345_anchored (the curve ids the exposure row names).

In [6]:
cy = yaml.safe_load(open(ROOT / "src/flood_lab/curves/curves.yaml"))
curves = {c["id"]: np.array(c["points"], dtype=float) for c in cy["substation_curves"] + cy["building_curves"]}

def interp(x, pts):
    xs, ys = pts[:, 0], pts[:, 1]
    if x < xs[0] or x > xs[-1]:
        raise ValueError(f"{x} outside {xs[0]}..{xs[-1]}")
    for i in range(len(xs) - 1):
        if xs[i] <= x <= xs[i + 1]:
            return ys[i] + (x - xs[i]) * (ys[i + 1] - ys[i]) / (xs[i + 1] - xs[i])

FAMILIES = {"low": "nirandjan_F2", "high": "williams_substation_max",
            "williams_min": "williams_substation_min", "bor": "bor_hv345_anchored"}
assert set(FAMILIES.values()) == set(ex.loc[SITE, "curve_id"].split("|"))
for k, cid in FAMILIES.items():
    print(k, cid, "domain", curves[cid][0, 0], curves[cid][-1, 0], "points", len(curves[cid]))
for b in ["B-3000013", "B-3000650"]:
    p = curves[ex.loc[b, "curve_id"]]
    print(b, ex.loc[b, "curve_id"], "domain", p[0, 0], p[-1, 0], "ratio at -4 ft", p[0, 1])

low nirandjan_F2 domain 0.0 6.0 points 63
high williams_substation_max domain 0.0 6.0 points 6
williams_min williams_substation_min domain 0.0 6.0 points 6
bor bor_hv345_anchored domain 0.0 6.0 points 601
B-3000013 hazus_struct_204 domain -1.2192 7.3152 ratio at -4 ft 0.0
B-3000650 hazus_struct_110 domain -1.2192 7.3152 ratio at -4 ft 0.03


In [7]:
def wet(d, cut):
    # cut = WET_M (builder's main case) or 0.0 (BD-1 as written: depth of 0 or less is dry)
    return d >= cut if cut > 0 else d > 0

def building_ratio(d, b, cut=WET_M):
    # returns (case, depth read, ratio)
    if not wet(d, cut):
        return "dry", np.nan, 0.0                           # BD-1
    ffh = ex.loc[b, "ffh_m"]
    if not np.isfinite(ffh):
        return "failed", np.nan, np.nan                     # BD-5
    pts = curves[ex.loc[b, "curve_id"]]
    x = d - ffh                                             # D-3, BD-2
    if x < pts[0, 0]:                                       # BD-3
        bsmt = ex.loc[b, "basement"]
        if bsmt == "Y":
            return "below -4 ft, basement", pts[0, 0], interp(pts[0, 0], pts)
        if bsmt == "N":
            return "below -4 ft, no basement", np.nan, 0.0
        return "failed", np.nan, np.nan
    if x > pts[-1, 0]:
        return "failed", np.nan, np.nan                     # BD-4
    return "read", x, interp(x, pts)

def substation_ratio(d, fam, cut=WET_M):
    if not wet(d, cut):
        return 0.0
    if d > 6.0:
        raise ValueError("C-6: depth above 6 m")
    return interp(d, curves[FAMILIES[fam]])

def per_run(cut):
    out = depth[["label", "weight"]].copy()
    for b in ["B-3000013", "B-3000650"]:
        r = [building_ratio(d, b, cut) for d in depth[b]]
        out[b + " depth"] = depth[b]
        out[b + " case"] = [x[0] for x in r]
        out[b + " read_m"] = [x[1] for x in r]
        out[b + " ratio"] = [x[2] for x in r]
        out[b + " loss"] = out[b + " ratio"] * ex.loc[b, "value_mid_usd_2026"]
    out[SITE + " depth"] = depth[SITE]
    for fam in FAMILIES:
        out[f"{SITE} {fam} ratio"] = [substation_ratio(d, fam, cut) for d in depth[SITE]]
        out[f"{SITE} {fam} loss"] = out[f"{SITE} {fam} ratio"] * ex.loc[SITE, "value_mid_usd_2026"]
    return out

main = per_run(WET_M)
alt = per_run(0.0)
for b in ["B-3000013", "B-3000650"]:
    print(b, main[b + " case"].value_counts().to_dict())

B-3000013 {'dry': 147, 'read': 3}
B-3000650 {'dry': 133, 'below -4 ft, basement': 17}


### Per-run table (main case, all 150 runs)

In [8]:
with pd.option_context("display.max_rows", 200, "display.float_format", "{:.6f}".format):
    display(main.drop(columns=[c for c in main.columns if c.endswith("read_m")]))

,label,weight,B-3000013 depth,B-3000013 case,B-3000013 ratio,B-3000013 loss,B-3000650 depth,B-3000650 case,B-3000650 ratio,B-3000650 loss,S-383102724 depth,S-383102724 low ratio,S-383102724 low loss,S-383102724 high ratio,S-383102724 high loss,S-383102724 williams_min ratio,S-383102724 williams_min loss,S-383102724 bor ratio,S-383102724 bor loss
0,cat_r0001,1.000000,0.140776,read,0.140540,140741.392375,0.437525,"below -4 ft, basement",0.030000,16716.503735,0.503000,0.033005,2058697.275805,0.334900,20889581.166680,0.143642,8959752.927266,0.032686,2038802.114428
1,cat_r0002,1.000000,0.163927,read,0.150129,150344.447564,0.438779,"below -4 ft, basement",0.030000,16716.503735,0.504000,0.033070,2062790.308478,0.335200,20908293.598895,0.143856,8973101.128913,0.032868,2050173.035737
2,cat_r0003,1.000000,0.030963,dry,0.000000,0.000000,0.324413,"below -4 ft, basement",0.030000,16716.503735,0.409000,0.026838,1674009.578789,0.273212,17041756.560756,0.116974,7296320.922719,0.017470,1089676.462728
3,cat_r0004,1.000000,0.038848,dry,0.000000,0.000000,0.356324,"below -4 ft, basement",0.030000,16716.503735,0.450000,0.029528,1841825.992105,0.300600,18750097.982499,0.128700,8027736.561369,0.023641,1474620.901414
4,cat_r0005,1.000000,0.009152,dry,0.000000,0.000000,0.332822,"below -4 ft, basement",0.030000,16716.503735,0.381000,0.025000,1559403.200141,0.254508,15875083.885704,0.108966,6796817.352262,0.013703,854707.681199
5,cat_r0006,1.000000,0.000000,dry,0.000000,0.000000,0.225336,"below -4 ft, basement",0.030000,16716.503735,0.152000,0.009974,622146.461209,0.101536,6333366.393771,0.043472,2711591.000926,0.000074,4615.791334
6,cat_r0007,1.000000,0.000000,dry,0.000000,0.000000,0.208896,"below -4 ft, basement",0.030000,16716.503735,0.148000,0.009712,605776.642559,0.098864,6166699.134857,0.042328,2640233.461930,0.000059,3705.109613
7,cat_r0008,1.000000,0.000000,dry,0.000000,0.000000,0.243371,"below -4 ft, basement",0.030000,16716.503735,0.165000,0.010827,675340.400196,0.110220,6875036.382232,0.047190,2943503.600776,0.000143,8919.710226
8,cat_r0009,1.000000,0.000000,dry,0.000000,0.000000,0.285177,"below -4 ft, basement",0.030000,16716.503735,0.166000,0.010893,679432.185374,0.110888,6916702.731297,0.047476,2961342.786154,0.000150,9331.383735
9,cat_r0010,1.000000,0.000000,dry,0.000000,0.000000,0.006591,dry,0.000000,0.000000,0.081000,0.005315,331539.915941,0.054108,3375017.736191,0.023166,1444992.623579,0.000000,0.000000


## 6. Risk figures (own computation)

* EAL = Σ w·L / 1000.
* CRC lower quantile at q = 0.99: the smallest loss L with Σ{w : loss ≤ L} / 1000 ≥ 0.99 (weights are
  whole years, so the test is done on integer sums: ≥ 990).
* Tools' convention: losses sorted descending, cumulative weight C_k, return period 1000 / C_k; the
  1-in-100 loss is the loss of the last rank whose return period is still ≥ 100 (C_k ≤ 10). Where no
  rank has C_k exactly 10 this is the convention's left end; the next rank (first with RP < 100) is
  printed beside it.

In [9]:
def figures(loss, w):
    loss = np.asarray(loss, float); w = np.asarray(w, float)
    eal = float((w * loss).sum() / M)
    o = np.argsort(loss, kind="stable")
    cum = np.cumsum(w[o])
    crc = float(loss[o][np.flatnonzero(cum >= 990)[0]])
    od = np.argsort(-loss, kind="stable")
    C = np.cumsum(w[od])
    k = np.flatnonzero(C <= 10)
    tools = float(loss[od][k[-1]]) if len(k) else np.nan
    nxt = float(loss[od][k[-1] + 1]) if len(k) else float(loss[od][0])
    return {"EAL": eal, "1in100 CRC": crc, "1in100 tools": tools,
            "C_k at tools rank": float(C[k[-1]]) if len(k) else np.nan, "next rank loss": nxt,
            "max": float(loss.max()), "wet runs": int((loss > 0).sum())}

def table(df):
    res = {}
    for b in ["B-3000013", "B-3000650"]:
        res[b] = figures(df[b + " loss"], df.weight)
    for fam in FAMILIES:
        res[f"{SITE} {fam}"] = figures(df[f"{SITE} {fam} loss"], df.weight)
    return pd.DataFrame(res).T

R = table(main)
R_alt = table(alt)
with pd.option_context("display.float_format", "{:,.6f}".format):
    print("MAIN CASE (depth < 0.05 m is 0)")
    print(R.to_string())
    print("\nBD-1 AS WRITTEN (only depth <= 0 is dry)")
    print(R_alt.to_string())
R.to_csv("/dev/null")

MAIN CASE (depth < 0.05 m is 0)
                                    EAL       1in100 CRC     1in100 tools  C_k at tools rank   next rank loss               max  wet runs
B-3000013                    390.320071         0.000000         0.000000          10.000000         0.000000    150,344.447564  3.000000
B-3000650                    300.897067    16,716.503735    16,716.503735          10.000000    16,716.503735     16,716.503735 17.000000
S-383102724 low           23,039.311501   675,340.400196   679,432.185374          10.000000   675,340.400196  2,062,790.308478 26.000000
S-383102724 high         234,382.219647 6,875,036.382232 6,916,702.731297          10.000000 6,875,036.382232 20,908,293.598895 26.000000
S-383102724 williams_min 100,386.630457 2,943,503.600776 2,961,342.786154          10.000000 2,943,503.600776  8,973,101.128913 26.000000
S-383102724 bor           10,355.331582     8,919.710226     9,331.383735          10.000000     8,919.710226  2,050,173.035737 16.000000

B

### Results table (own numbers, main case)

Written down here (2026-11-28) from the outputs above, before any file in `site/flood/data/f6/` was opened.
Dollars of 2026 (4Q26), mid values, year weights = F4 block weights (sum 1,000), depths < 0.05 m count as 0.

| Asset | EAL (USD/yr) | 1-in-100, CRC lower quantile | 1-in-100, tools (RP = 1000/C_k) | runs with loss |
|---|---:|---:|---:|---:|
| B-3000013 (RES3D, ffh 0.16 m, no basement, curve 204) | 390.320071 | 0.00 | 0.00 | 3 |
| B-3000650 (RES1, ffh 3.844 m, basement Y, curve 110, −4 ft read = 0.03) | 300.897067 | 16,716.503735 | 16,716.503735 | 17 |
| S-383102724 low (Nirandjan F2) | 23,039.311501 | 675,340.400196 | 679,432.185374 | 26 |
| S-383102724 high (Williams substation max) | 234,382.219647 | 6,875,036.382232 | 6,916,702.731297 | 26 |
| S-383102724 williams_min | 100,386.630457 | 2,943,503.600776 | 2,961,342.786154 | 26 |
| S-383102724 bor (HV345 anchored) | 10,355.331582 | 8,919.710226 | 9,331.383735 | 16 |

The tools' rank has cumulative weight exactly 10 for every asset (the ten heaviest losses all fall in T1
runs of weight 1), so the tools' 1-in-100 is the 10th largest and CRC's the 11th.

With BD-1 as the rules page writes it (dry only at depth ≤ 0), EALs would be B-3000013 715.290069,
B-3000650 401.196090, substation low 28,098.365054, high 285,882.490497, williams_min 122,436.147618,
bor 10,355.331582 (Bor is 0 below 0.1 m anyway); the 1-in-100 figures do not change.

## 7. Comparison with the builder's outputs (`site/flood/data/f6/`)

In [10]:
F6 = ROOT / "site/flood/data/f6"
b_bld = pd.read_csv(F6 / "buildings.csv")
b_sub = pd.read_csv(F6 / "substations.csv")
b_ev = pd.read_csv(F6 / "substation_event_losses.csv")
metrics = json.load(open(F6 / "metrics.json"))
print(b_bld.columns.tolist()); print(b_sub.columns.tolist()); print(b_ev.columns.tolist())
print(list(metrics)[:40])

['LocNumber', 'curve_id', 'ffh_m', 'basement', 'value_mid_usd_2026', 'value_substitute', 'lot_shared_with_nsi', 'runs_read', 'runs_no_loss', 'runs_failed', 'failure', 'eal_event_table', 'eal_point', 'eal_grid', 'depth_10y_point', 'loss_10y_point', 'depth_100y_point', 'loss_100y_point', 'depth_500y_point', 'loss_500y_point', 'depth_10y_grid', 'loss_10y_grid', 'depth_100y_grid', 'loss_100y_grid', 'depth_500y_grid', 'loss_500y_grid']
['LocNumber', 'name', 'family', 'curve_id', 'value_mid_usd_2026', 'eal_event_table', 'ead_point', 'eal_point', 'damage_mean_pushforward_point', 'damage_q10_pushforward_point', 'damage_q100_pushforward_point', 'damage_q500_pushforward_point', 'eal_grid', 'grid_failure', 'depth_10y_point', 'ratio_10y_point', 'loss_10y_point', 'depth_10y_grid', 'ratio_10y_grid', 'loss_10y_grid', 'failure_10y_grid', 'depth_100y_point', 'ratio_100y_point', 'loss_100y_point', 'depth_100y_grid', 'ratio_100y_grid', 'loss_100y_grid', 'failure_100y_grid', 'depth_500y_point', 'ratio_500

In [11]:
print(b_bld[b_bld.LocNumber.isin(["B-3000013", "B-3000650"])].T.to_string())
print(b_sub[b_sub.LocNumber == SITE].T.to_string())

                                  3                 340
LocNumber                   B-3000013         B-3000650
curve_id             hazus_struct_204  hazus_struct_110
ffh_m                            0.16             3.844
basement                            N                 Y
value_mid_usd_2026      1001436.09036     557216.791154
value_substitute                False              True
lot_shared_with_nsi             False             False
runs_read                           3                17
runs_no_loss                      147               133
runs_failed                         0                 0
failure                           NaN               NaN
eal_event_table            390.320071        300.897067
eal_point                  390.320109        300.897068
eal_grid                          NaN               NaN
depth_10y_point                   0.0               0.0
loss_10y_point                    0.0               0.0
depth_100y_point                  0.0          0

In [12]:
def close(a, b):
    a, b = float(a), float(b)
    return abs(a - b) < 0.005 and (abs(a - b) <= 1e-9 * max(abs(a), abs(b)) or a == b)

def quantiles(loss, w, T):
    # own figures at other return periods, for the builder's 10/100/500 y columns
    loss = np.asarray(loss, float); w = np.asarray(w, float)
    o = np.argsort(loss, kind="stable"); cum = np.cumsum(w[o])
    crc = float(loss[o][np.flatnonzero(cum >= M - M / T)[0]])
    od = np.argsort(-loss, kind="stable"); C = np.cumsum(w[od]); k = np.flatnonzero(C <= M / T)
    tools = float(loss[od][k[-1]]) if len(k) else 0.0
    return crc, tools

diffs, checks = [], 0
def check(what, mine, theirs):
    global checks
    checks += 1
    if not close(mine, theirs):
        diffs.append((what, mine, theirs, float(mine) - float(theirs)))

# runs weights against annual_losses.csv
al = pd.read_csv(F6 / "annual_losses.csv").set_index("event_id")
for r in runs.itertuples():
    check(f"mass {r.label}", r.weight / M, al.loc[r.label, "mass"])

# per run, per family: depth used, ratio, loss
ev = b_ev[b_ev.LocNumber == SITE].set_index(["event_id", "family"])
assert len(ev) == 600
for r in main.itertuples(index=False):
    row = dict(zip(main.columns, r))
    for fam in FAMILIES:
        e = ev.loc[(row["label"], fam)]
        assert e.curve_id == FAMILIES[fam]
        check(f"{row['label']} {fam} depth_raw", depth.set_index('label').loc[row['label'], SITE], e.depth_raw_m)
        used = row[SITE + " depth"] if row[SITE + " depth"] >= WET_M else 0.0
        check(f"{row['label']} {fam} depth_used", used, e.depth_used_m)
        check(f"{row['label']} {fam} ratio", row[f"{SITE} {fam} ratio"], e.damage_ratio)
        check(f"{row['label']} {fam} loss", row[f"{SITE} {fam} loss"], e.loss_usd_2026)

out = []
sub = b_sub[b_sub.LocNumber == SITE].set_index("family")
for fam in FAMILIES:
    s = sub.loc[fam]; key = f"{SITE} {fam}"
    check(f"{key} value", ex.loc[SITE, "value_mid_usd_2026"], s.value_mid_usd_2026)
    check(f"{key} eal_event_table", R.loc[key, "EAL"], s.eal_event_table)
    row = {"asset": key, "EAL mine": R.loc[key, "EAL"], "EAL builder": s.eal_event_table,
           "eal_point (builder, CRC expectation)": s.eal_point}
    for T in (10, 100, 500):
        crc, tools = quantiles(main[f"{key} loss"], main.weight, T)
        row[f"{T}y CRC mine"], row[f"{T}y tools mine"], row[f"loss_{T}y_point"] = crc, tools, s[f"loss_{T}y_point"]
    out.append(row)
bb = b_bld.set_index("LocNumber")
for b in ["B-3000013", "B-3000650"]:
    s = bb.loc[b]
    check(f"{b} value", ex.loc[b, "value_mid_usd_2026"], s.value_mid_usd_2026)
    check(f"{b} eal_event_table", R.loc[b, "EAL"], s.eal_event_table)
    read = int((main[b + " case"] != "dry").sum())
    check(f"{b} runs_read", read, s.runs_read)
    check(f"{b} runs_no_loss", 150 - read, s.runs_no_loss)
    check(f"{b} runs_failed", int((main[b + " case"] == "failed").sum()), s.runs_failed)
    row = {"asset": b, "EAL mine": R.loc[b, "EAL"], "EAL builder": s.eal_event_table,
           "eal_point (builder, CRC expectation)": s.eal_point}
    for T in (10, 100, 500):
        crc, tools = quantiles(main[b + " loss"], main.weight, T)
        row[f"{T}y CRC mine"], row[f"{T}y tools mine"], row[f"loss_{T}y_point"] = crc, tools, s[f"loss_{T}y_point"]
    out.append(row)
O = pd.DataFrame(out).set_index("asset")
with pd.option_context("display.float_format", "{:,.6f}".format):
    print(O[["EAL mine", "EAL builder", "eal_point (builder, CRC expectation)"]].to_string())
    print()
    print(O[[c for c in O.columns if "y " in c or c.startswith("loss_")]].T.to_string())
print(f"\n{checks} checks to the cent and 1e-9 relative; {len(diffs)} differences")
for d in diffs:
    print(d)
# which convention do the builder's loss_Ty_point columns follow?
for a in O.index:
    for T in (10, 100, 500):
        m = O.loc[a]
        print(a, T, "y: point =", "tools" if close(m[f"{T}y tools mine"], m[f"loss_{T}y_point"]) else "",
              "CRC" if close(m[f"{T}y CRC mine"], m[f"loss_{T}y_point"]) else "")

                               EAL mine    EAL builder  eal_point (builder, CRC expectation)
asset                                                                                       
S-383102724 low           23,039.311501  23,039.311501                         23,039.312532
S-383102724 high         234,382.219647 234,382.219647                        234,382.230133
S-383102724 williams_min 100,386.630457 100,386.630457                        100,386.634950
S-383102724 bor           10,355.331582  10,355.331582                         10,355.332454
B-3000013                    390.320071     390.320071                            390.320109
B-3000650                    300.897067     300.897067                            300.897068

asset            S-383102724 low  S-383102724 high  S-383102724 williams_min  S-383102724 bor      B-3000013     B-3000650
10y CRC mine            0.000000          0.000000                  0.000000         0.000000       0.000000      0.000000
10y tools


## 8. Findings

**Match.** Every compared figure agrees to the cent and to 1e-9 relative (2,568 checks, 0 differences):
the 150 run masses; for the substation, 600 rows × raw depth, depth used, damage ratio and loss; each
asset's mid value and `eal_event_table`; the buildings' `runs_read`, `runs_no_loss`, `runs_failed`; and
`loss_{10,100,500}y_point` against the tools' convention.

**Differences and their causes (none in the main figures):**

1. `loss_{T}y_point` (and `depth_{T}y_point`) follow the **tools' convention** (k-th largest with
   cumulative weight M/T, i.e. the catalogue's "largest d with P(D ≥ d) ≥ p"), not CRC's lower quantile:
   at 1-in-100 the substation's is the 10th largest (low $679,432.19), CRC's would be the 11th
   ($675,340.40); at 1-in-500 B-3000013 is $140,741.39 (tools) against $99,234.23 (CRC). `F6-risk.md`
   (section "Hand recomputation") describes these columns as "depth at T in CRC's convention on the step
   curve": the label does not match the numbers. The builder writes no per-asset CRC-convention 1-in-100
   for these assets, so the CRC column above has no counterpart to compare.
2. `eal_point` differs from Σ w·L / 1000 by up to $0.0105 (high family, 4.5e-8 relative). It is a
   separate route (an expectation over the depth distribution computed in CRC, not seen here); the event
   table EAL, the figure asked for, matches exactly.

**Rules page against the builder's convention:**

* The 0.05 m dry cut (`catalogue.WET_M`) is not in `loss-rules.md`. BD-1 says only a depth of 0 or less is
  dry, and rule C has no dry threshold for substations. With BD-1 as written, EAL rises from 390.32 to
  715.29 (B-3000013), 300.90 to 401.20 (B-3000650), and by 22% for the substation's low, high and
  Williams-min families (Bor unchanged, 0 below 0.1 m); the 1-in-100 figures do not change. The builder's
  `results.dry_threshold_0_literal_bd1` sensitivity exists; the main case should cite a rule once Yuri decides.
* Hudson Avenue (138 kV, distribution) takes `bor_hv345_anchored` (Bor's HV > 345 kV row); C-3 names
  "Bor 2026" without a voltage row, so this choice is not stated by a rule.
